## Classifier improvements

compare the baseline AdaBoost from `bdt.ipynb` with histogram-based gradient boosting. use 10-fold stratified cross-validation to get a robust accuracy estimate, then run a Punzi scan on the improved model and write its efficiencies back into `bdt_results.json`.

no engineered features - same 7 raw inputs (MASS excluded).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import pickle
import os
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.ensemble import AdaBoostClassifier, HistGradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc

os.makedirs("../plots", exist_ok=True)

In [ ]:
features  = ["PT1", "PT2", "P1", "P2", "TotalPT", "VertexChisq", "Isolation", "MASS"]
features7 = [f for f in features if f != "MASS"]

signal = pd.read_csv("../data/signal_Bs2MuMu.txt", sep=r"\s+", header=None, names=features)
background = pd.read_csv("../data/background_combinatorial.txt", sep=r"\s+", header=None, names=features)
signal = signal.apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)
background = background.apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)

X = pd.concat([signal[features7], background[features7]], axis=0).reset_index(drop=True)
y = np.concatenate([np.ones(len(signal)), np.zeros(len(background))])

In [ ]:
def make_adaboost():
    base = DecisionTreeClassifier(max_depth=2, min_samples_leaf=50, random_state=42)
    return AdaBoostClassifier(estimator=base, n_estimators=100, learning_rate=0.5, random_state=42)

def make_gradboost():
    return HistGradientBoostingClassifier(max_depth=4, max_iter=200, learning_rate=0.1,
                                          min_samples_leaf=50, random_state=42)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_results = {}
for name, factory in [("AdaBoost", make_adaboost), ("GradBoost", make_gradboost)]:
    accs = []
    for train_idx, test_idx in cv.split(X, y):
        clf = factory()
        clf.fit(X.iloc[train_idx], y[train_idx])
        accs.append(accuracy_score(y[test_idx], clf.predict(X.iloc[test_idx])))
    cv_results[name] = {"folds": [float(a) for a in accs],
                        "mean": float(np.mean(accs)),
                        "std":  float(np.std(accs))}
    print(f"{name:10s}: {np.mean(accs):.4f} +/- {np.std(accs):.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
names = list(cv_results.keys())
means = [cv_results[n]["mean"] for n in names]
stds  = [cv_results[n]["std"]  for n in names]
x = np.arange(len(names))
ax.bar(x, means, yerr=stds, capsize=6, color=["steelblue", "darkorange"])
for xi, m, s in zip(x, means, stds):
    ax.text(xi, m + s + 0.003, f"{m:.4f}\n+/- {s:.4f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylim(min(means) - 0.03, max(means) + 0.02)
ax.set_ylabel("10-fold CV accuracy")
plt.tight_layout()
plt.savefig("../plots/cv_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
test_metrics = {}
fitted = {}
roc_data = {}
for name, factory in [("AdaBoost", make_adaboost), ("GradBoost", make_gradboost)]:
    clf = factory()
    clf.fit(X_train, y_train)
    fitted[name] = clf
    y_pred  = clf.predict(X_test)
    y_score = clf.predict_proba(X_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    test_metrics[name] = {"accuracy": float((tp+tn)/(tp+tn+fp+fn)),
                          "sig_eff":  float(tp/(tp+fn)),
                          "bkg_eff":  float(fp/(fp+tn))}
    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_data[name] = {"fpr": fpr, "tpr": tpr, "auc": float(auc(fpr, tpr))}
    print(f"{name:10s} test: acc={test_metrics[name]['accuracy']:.4f}  "
          f"sig_eff={test_metrics[name]['sig_eff']:.4f}  bkg_eff={test_metrics[name]['bkg_eff']:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for name, c in zip(names, ["steelblue", "darkorange"]):
    d = roc_data[name]
    ax.plot(d["fpr"], d["tpr"], color=c, label=f"{name} (AUC = {d['auc']:.4f})")
ax.plot([0,1],[0,1], "k--", lw=0.8)
ax.set_xlabel("background efficiency"); ax.set_ylabel("signal efficiency"); ax.legend()
plt.tight_layout()
plt.savefig("../plots/roc_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# overtraining check: training vs test score distributions for the improved model
fig, ax = plt.subplots(figsize=(7, 4))
clf = fitted["GradBoost"]
bins = np.linspace(0, 1, 40)
for label, Xs, ys, htype, alpha in [("train", X_train, y_train, "stepfilled", 0.4),
                                     ("test",  X_test,  y_test,  "step",       1.0)]:
    ss = clf.predict_proba(Xs[ys == 1])[:, 1]
    sb = clf.predict_proba(Xs[ys == 0])[:, 1]
    ax.hist(ss, bins=bins, density=True, alpha=alpha, histtype=htype, color="steelblue",
            label=f"signal {label}")
    ax.hist(sb, bins=bins, density=True, alpha=alpha, histtype=htype, color="darkorange",
            label=f"background {label}")
ax.set_xlabel("GradBoost score"); ax.set_ylabel("density"); ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.savefig("../plots/overtraining_check.png", dpi=150, bbox_inches="tight")
plt.show()

### Punzi scan on the improved model

run the same FOM scan used in `punzi_fom.ipynb`, but on GradBoost. write the resulting efficiencies and threshold back to `bdt_results.json` so `mass_fit.ipynb` can pick them up.

In [ ]:
gb = fitted["GradBoost"]
scores_sig = gb.predict_proba(signal[features7])[:, 1]
scores_bkg = gb.predict_proba(background[features7])[:, 1]

N_BKG_YEAR = 2000
n_sigma = 5.0
bkg_eff_floor = 1.0 / len(background)

thresholds = np.linspace(0.01, 0.99, 300)
fom_vals, sig_eff_scan, bkg_eff_scan = [], [], []
for t in thresholds:
    eps_s = (scores_sig >= t).mean()
    eps_b = max((scores_bkg >= t).mean(), bkg_eff_floor)
    B = N_BKG_YEAR * eps_b
    fom_vals.append(eps_s / (n_sigma/2 + np.sqrt(B)))
    sig_eff_scan.append(eps_s); bkg_eff_scan.append(eps_b)
fom_vals = np.array(fom_vals); sig_eff_scan = np.array(sig_eff_scan); bkg_eff_scan = np.array(bkg_eff_scan)

i_fom = int(np.argmax(fom_vals))
thr_punzi_imp = float(thresholds[i_fom])
sig_eff_imp_punzi = float(sig_eff_scan[i_fom])
bkg_eff_imp_punzi = float(bkg_eff_scan[i_fom])
print(f"improved Punzi: t={thr_punzi_imp:.3f}  sig_eff={sig_eff_imp_punzi:.4f}  bkg_eff={bkg_eff_imp_punzi:.4f}")

# comparison plot vs baseline AdaBoost Punzi (read from bdt_results.json)
try:
    with open("../bdt_results.json") as fh:
        bdt_res = json.load(fh)
    base_thr = bdt_res.get("punzi_threshold")
    base_sig = bdt_res.get("signal_efficiency_punzi")
    base_bkg = bdt_res.get("background_efficiency_punzi")
except Exception:
    base_thr = base_sig = base_bkg = None

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
ax = axes[0]
ax.plot(thresholds, fom_vals, color="steelblue", label="GradBoost Punzi FOM")
ax.axvline(thr_punzi_imp, color="steelblue", linestyle=":", alpha=0.8)
ax.set_xlabel("GradBoost threshold"); ax.set_ylabel("Punzi FOM")
ax.legend()

ax = axes[1]
ax.plot(thresholds, sig_eff_scan, label="signal")
ax.plot(thresholds, bkg_eff_scan, label="background")
ax.axvline(thr_punzi_imp, linestyle="--", color="black", label="improved Punzi")
ax.set_xlabel("threshold"); ax.set_ylabel("efficiency"); ax.legend()

plt.tight_layout()
plt.savefig("../plots/punzi_improved.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# save model + per-classifier results
with open("../improved_model.pkl", "wb") as fh:
    pickle.dump(gb, fh)

improved = {
    "classifier": "HistGradientBoostingClassifier",
    "features": features7,
    "cv_results": cv_results,
    "test_metrics": test_metrics,
    "signal_efficiency_improved": float(signal[features7].pipe(lambda d: gb.predict(d).mean())),
    "background_efficiency_improved": float(background[features7].pipe(lambda d: gb.predict(d).mean())),
}
with open("../improved_results.json", "w") as fh:
    json.dump(improved, fh, indent=2)

# also write improved Punzi working point into bdt_results.json
with open("../bdt_results.json") as fh:
    bdt_res = json.load(fh)
bdt_res["signal_efficiency_improved_punzi"]     = sig_eff_imp_punzi
bdt_res["background_efficiency_improved_punzi"] = bkg_eff_imp_punzi
bdt_res["improved_punzi_threshold"]             = thr_punzi_imp
with open("../bdt_results.json", "w") as fh:
    json.dump(bdt_res, fh, indent=2)
print("saved improved_model.pkl, improved_results.json, and updated bdt_results.json")